In [1]:
import os
import ast
import hashlib
import re
from hashlib import md5

In [17]:
import requests
import subprocess
import time
import random

In [5]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing

In [3]:
print(os.getcwd())

d:\Skills\Machine_Learning\GPT\notebook


In [ ]:
# -------------------------------
# CONFIG
# -------------------------------
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DIR = os.path.join(BASE_DIR, "Data/raw")

GITHUB_API = "https://api.github.com/search/repositories"

GITHUB_TOKEN = ""  # add token if you hit rate limits

HEADERS = {
    "Accept": "application/vnd.github+json"
}

if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"

# search queries (HIGH QUALITY)
QUERIES = [
    #"language:python stars:>500",
    #"language:python stars:>1000",
    "machine learning python stars:>200",
    "data structures python stars:>200",
]

MAX_REPOS = 300   # target repos


# -------------------------------
# CREATE RAW DIR
# -------------------------------
os.makedirs(RAW_DIR, exist_ok=True)


# -------------------------------
# GET REPOS FROM GITHUB
# -------------------------------
def fetch_repos():
    repo_urls = set()

    for query in QUERIES:
        print(f"\n🔍 Searching: {query}")

        for page in range(1, 6):  # pagination (max ~1000 results)
            params = {
                "q": query,
                "sort": "stars",
                "order": "desc",
                "per_page": 30,
                "page": page
            }

            response = requests.get(GITHUB_API, headers=HEADERS, params=params)

            if response.status_code != 200:
                print("❌ API Error:", response.json())
                continue

            data = response.json()

            for item in data.get("items", []):
                repo_urls.add(item["clone_url"])

                if len(repo_urls) >= MAX_REPOS:
                    return list(repo_urls)

            time.sleep(1)  # avoid rate limit

    return list(repo_urls)


# -------------------------------
# CLONE REPOS
# -------------------------------
def clone_repo(repo_url):
    name = repo_url.split("/")[-1].replace(".git", "")
    path = os.path.join(RAW_DIR, name)

    if os.path.exists(path):
        print(f"⏭️ Skipping (already exists): {name}")
        return

    try:
        print(f"⬇️ Cloning: {name}")
        subprocess.run(
            ["git", "clone", "--depth", "1", repo_url, path],
            check=True
        )
    except subprocess.CalledProcessError:
        print(f"Failed: {repo_url}")


# -------------------------------
# MAIN
# -------------------------------
def main():
    print("Fetching repositories...")
    repos = fetch_repos()

    print(f"\nFound {len(repos)} repositories\n")

    for repo in repos:
        clone_repo(repo)

    print("\nDone! All repos saved to Data/raw")


if __name__ == "__main__":
    main()

In [ ]:
import os
import ast
import hashlib
import multiprocessing
from concurrent.futures import ProcessPoolExecutor

# ==============================
# PATHS
# ==============================
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

RAW_DIR = os.path.join(BASE_DIR, "Data/raw")
OUTPUT_FILE = os.path.join(BASE_DIR, "Data/processed/code_11.txt")

MIN_LINES = 5
MAX_LINES = 50


# ==============================
# EXTRACT BLOCKS
# ==============================
def extract_blocks(source):
    try:
        tree = ast.parse(source)
        return [
            ast.unparse(node)
            for node in tree.body
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))
        ]
    except:
        return []


# ==============================
# CLEAN CODE
# ==============================
def clean_code(code):
    cleaned = []
    for line in code.split("\n"):
        if line.strip().startswith("#"):
            continue
        if "#" in line:
            line = line.split("#")[0]
        if line.strip():
            cleaned.append(line.rstrip())
    return "\n".join(cleaned)


# ==============================
# FILTERS
# ==============================
BAD_LIBS = ["tkinter","pygame","kivy","requests","cv2","selenium"]
BAD_LIBS_ULTRA = ["numpy","pandas","csv.","smtplib"]

def is_ui_or_external(code):
    return any(lib in code for lib in BAD_LIBS)

def is_ultra_bad(code):
    return any(lib in code for lib in BAD_LIBS_ULTRA)

def is_bad_block(code):
    return any(k in code for k in ["input(","print(","unittest","assert "])

def has_suspicious_code(code):
    return any(p in code for p in ["undefined","except:"])

def is_useless(code):
    return any(p in code.lower() for p in ["todo","example","demo"])

def has_side_effects(code):
    return any(p in code for p in [".save(", ".write(", ".show("])

def has_many_imports(code):
    return code.count("import ") > 3

def is_too_complex(code):
    return len(code.split("\n")) > 40

def is_game_code(code):
    return any(k in code for k in ["Game","Player","Grid"])


# ==============================
# VALIDATION + HASH
# ==============================
def is_valid(code):
    try:
        ast.parse(code)
        return True
    except:
        return False

def normalize(code):
    return "\n".join(line.strip() for line in code.split("\n"))

def hash_code(code):
    return hashlib.md5(code.encode()).hexdigest()


# ==============================
# PROCESS SINGLE FILE (SAFE)
# ==============================
def process_file(path):
    local_seen = set()
    local_blocks = []
    local_names = set()

    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            code = f.read()

        blocks = extract_blocks(code)

        for block in blocks:
            try:
                block = clean_code(block)

                lines = len(block.split("\n"))
                if lines < MIN_LINES or lines > MAX_LINES:
                    continue

                if is_bad_block(block): continue
                if is_ui_or_external(block): continue
                if is_ultra_bad(block): continue
                if has_suspicious_code(block): continue
                if is_useless(block): continue
                if has_side_effects(block): continue
                if has_many_imports(block): continue
                if is_too_complex(block): continue
                if is_game_code(block): continue

                # duplicate name (local only)
                try:
                    tree = ast.parse(block)
                    for node in tree.body:
                        if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
                            if node.name in local_names:
                                raise Exception
                            local_names.add(node.name)
                except:
                    continue

                if not is_valid(block):
                    continue

                norm = normalize(block)
                h = hash_code(norm)

                if h in local_seen:
                    continue

                local_seen.add(h)
                local_blocks.append(block)

            except:
                continue

    except:
        return []

    return local_blocks


# ==============================
# MAIN PIPELINE (PARALLEL SAFE)
# ==============================
def process():
    all_files = [
        os.path.join(root, file)
        for root, _, files in os.walk(RAW_DIR)
        for file in files if file.endswith(".py")
    ]

    print(f"Total files: {len(all_files)}")

    final_blocks = []
    global_seen = set()

    # safer worker count
    num_workers = max(1, multiprocessing.cpu_count() - 1)

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        for result in executor.map(process_file, all_files, chunksize=1000):

            for block in result:
                norm = normalize(block)
                h = hash_code(norm)

                if h in global_seen:
                    continue

                global_seen.add(h)
                final_blocks.append(block)

    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write("\n\n".join(final_blocks))

    print("===================================")
    print(f"Final blocks: {len(final_blocks)}")
    print(f"Saved to: {OUTPUT_FILE}")
    print("===================================")


# ==============================
# ENTRY POINT (WINDOWS FIX)
# ==============================
if __name__ == "__main__":
    multiprocessing.set_start_method("spawn", force=True)
    process()

Total files: 122418


In [3]:
import os
import ast
import hashlib

# ==============================
# PATHS
# ==============================
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

RAW_DIR = os.path.join(BASE_DIR, "Data/raw")
OUTPUT_FILE = os.path.join(BASE_DIR, "Data/processed/code_11.txt")

MIN_LINES = 5
MAX_LINES = 50

# ==============================
# FILTER CONFIG (FAST)
# ==============================
BAD_KEYWORDS = (
    "input(", "print(", "unittest", "assert ",
    "tkinter", "pygame", "kivy", "selenium",
    "cv2", "requests", "numpy", "pandas",
    "todo", "example", "demo",
    ".save(", ".write(", ".show(",
    "undefined", "except:"
)

GAME_KEYWORDS = ("Game", "Player", "Grid")


# ==============================
# FAST BLOCK EXTRACTION
# ==============================
def extract_blocks(source):
    try:
        tree = ast.parse(source)
    except:
        return []

    lines = source.splitlines()
    blocks = []

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            try:
                start = node.lineno - 1
                end = node.end_lineno
                blocks.append("\n".join(lines[start:end]))
            except:
                continue

    return blocks


# ==============================
# FAST FILTER
# ==============================
def is_valid_block(code):
    line_count = code.count("\n") + 1

    if line_count < MIN_LINES or line_count > MAX_LINES:
        return False

    lowered = code.lower()

    for k in BAD_KEYWORDS:
        if k in lowered:
            return False

    for k in GAME_KEYWORDS:
        if k in code:
            return False

    if code.count("import ") > 3:
        return False

    return True


# ==============================
# MAIN PIPELINE
# ==============================
def process():
    global_seen = set()
    processed = 0
    total_files = 0

    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as out:

        for root, _, files in os.walk(RAW_DIR):
            for file in files:
                if not file.endswith(".py"):
                    continue

                total_files += 1
                path = os.path.join(root, file)

                try:
                    # skip large files (huge speed boost)
                    if os.path.getsize(path) > 500_000:  # 500 KB
                        continue

                    with open(path, "r", encoding="utf-8", errors="ignore") as f:
                        source = f.read()

                except:
                    continue

                blocks = extract_blocks(source)

                for block in blocks:
                    try:
                        if not is_valid_block(block):
                            continue

                        # validate syntax once
                        try:
                            ast.parse(block)
                        except:
                            continue

                        norm = "\n".join(line.strip() for line in block.split("\n"))
                        h = hashlib.md5(norm.encode()).hexdigest()

                        if h in global_seen:
                            continue

                        global_seen.add(h)
                        out.write(block + "\n\n")

                    except:
                        continue

                processed += 1

                if processed % 1000 == 0:
                    print(f"Processed {processed} files")

    print("===================================")
    print(f"Total files scanned: {total_files}")
    print(f"Unique blocks: {len(global_seen)}")
    print(f"Saved to: {OUTPUT_FILE}")
    print("===================================")


# ==============================
# ENTRY POINT
# ==============================
if __name__ == "__main__":
    process()

Processed 1000 files
Processed 2000 files
Processed 3000 files
Processed 4000 files
Processed 5000 files
Processed 6000 files
Processed 7000 files
Processed 8000 files
Processed 9000 files
Processed 10000 files
Processed 11000 files
Processed 12000 files
Processed 13000 files
Processed 14000 files
Processed 15000 files
Processed 16000 files
Processed 17000 files
Processed 18000 files
Processed 19000 files
Processed 20000 files
Processed 21000 files
Processed 22000 files
Processed 23000 files
Processed 24000 files
Processed 25000 files
Processed 26000 files
Processed 27000 files
Processed 28000 files
Processed 29000 files
Processed 30000 files
Processed 31000 files
Processed 32000 files
Processed 33000 files
Processed 34000 files
Processed 35000 files
Processed 36000 files
Processed 37000 files
Processed 38000 files
Processed 39000 files
Processed 40000 files
Processed 41000 files
Processed 42000 files
Processed 43000 files
Processed 44000 files
Processed 45000 files
Processed 46000 fil

In [3]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "code_11.txt")
OUTPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "clean_code_2.txt")


# ==============================
# 1. REMOVE NOISE
# ==============================
def remove_noise(code: str) -> str:
    cleaned_lines = []

    for line in code.splitlines():
        line_strip = line.strip()

        # Remove separators / junk
        if "=====" in line_strip:
            continue
        if line_strip.startswith("# ====="):
            continue

        # Remove excessive empty lines
        if line_strip == "" and (len(cleaned_lines) > 0 and cleaned_lines[-1] == ""):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


# ==============================
# 2. SPLIT INTO FUNCTIONS/CLASSES
# ==============================
def split_into_samples(code: str):
    pattern = r"(def\s+\w+\(.*?\):|class\s+\w+\(?.*?\)?:)"
    matches = list(re.finditer(pattern, code))

    samples = []

    for i in range(len(matches)):
        start = matches[i].start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(code)

        snippet = code[start:end].strip()
        samples.append(snippet)

    return samples


# ==============================
# 3. VALIDATE CODE (REMOVE BROKEN)
# ==============================
def is_valid_python(code: str) -> bool:
    try:
        ast.parse(code)
        return True
    except:
        return False


def is_useful_sample(code: str) -> bool:
    # Remove useless or harmful samples
    if len(code) < 30:
        return False

    if "NotImplementedError" in code:
        return False

    if re.search(r"\bpass\b", code) and "class" in code:
        return False

    return True


def filter_samples(samples):
    valid = []
    for s in samples:
        if is_valid_python(s) and is_useful_sample(s):
            valid.append(s)
    return valid


# ==============================
# 4. DEDUPLICATION
# ==============================
def deduplicate(samples):
    seen = set()
    unique = []

    for s in samples:
        h = md5(s.encode()).hexdigest()
        if h not in seen:
            seen.add(h)
            unique.append(s)

    return unique


# ==============================
# 5. SAVE DATASET
# ==============================
def save_samples(samples, output_file):
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, "w", encoding="utf-8") as f:
        for s in samples:
            f.write(s.strip())
            f.write("\n\n# ===== SAMPLE SEPARATOR =====\n\n")


# ==============================
# MAIN PIPELINE
# ==============================
def process():
    print("Loading data...")

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        raw_code = f.read()

    print("Removing noise...")
    clean_code = remove_noise(raw_code)

    print("Splitting into samples...")
    samples = split_into_samples(clean_code)

    print(f"Total samples before filtering: {len(samples)}")

    print("Filtering broken/useless samples...")
    samples = filter_samples(samples)

    print(f"After filtering: {len(samples)}")

    print("Deduplicating...")
    samples = deduplicate(samples)

    print(f"After deduplication: {len(samples)}")

    print("Saving dataset...")
    save_samples(samples, OUTPUT_FILE)

    print("Done")


if __name__ == "__main__":
    process()

Loading data...
Removing noise...
Splitting into samples...
Total samples before filtering: 223080
Filtering broken/useless samples...
After filtering: 170201
Deduplicating...
After deduplication: 159551
Saving dataset...
Done


In [4]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "code_11.txt")
OUTPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "clean_code_3.txt")


# ==============================
# 1. REMOVE NOISE
# ==============================
def remove_noise(code: str) -> str:
    cleaned_lines = []

    for line in code.splitlines():
        line_strip = line.strip()

        # Remove separators / junk
        if "=====" in line_strip:
            continue
        if line_strip.startswith("# ====="):
            continue

        # Remove excessive empty lines
        if line_strip == "" and (len(cleaned_lines) > 0 and cleaned_lines[-1] == ""):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


# ==============================
# 2. HYBRID SPLITTING (AST + FALLBACK) 
# ==============================
def extract_code_blocks(code: str):
    blocks = []

    # Try full AST parse first
    try:
        tree = ast.parse(code)

        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                try:
                    block = ast.get_source_segment(code, node)
                    if block:
                        blocks.append(block.strip())
                except:
                    continue

        if blocks:
            return blocks

    except:
        pass  # fallback

    # ==============================
    # FALLBACK: REGEX + AST VALIDATION
    # ==============================
    pattern = r"(def\s+\w+\(.*?\):|class\s+\w+\(?.*?\)?:)"
    matches = list(re.finditer(pattern, code))

    for i in range(len(matches)):
        start = matches[i].start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(code)

        snippet = code[start:end].strip()

        try:
            ast.parse(snippet)
            blocks.append(snippet)
        except:
            continue

    return blocks


# ==============================
# 3. FILTER BAD SAMPLES
# ==============================
def is_valid_sample(code: str) -> bool:
    try:
        tree = ast.parse(code)
    except:
        return False

    if not tree.body:
        return False

    node = tree.body[0]

    # Remove empty classes
    if isinstance(node, ast.ClassDef):
        if len(node.body) < 2:
            return False

    # Remove trivial functions
    if isinstance(node, ast.FunctionDef):
        if len(node.body) < 2:
            return False

    # Remove too small code
    if len(code.splitlines()) < 3:
        return False

    # Remove pass-only junk
    if "pass" in code and len(code.splitlines()) < 10:
        return False

    return True


def filter_samples(samples):
    return [s for s in samples if is_valid_sample(s)]


# ==============================
# 4. DEDUPLICATION
# ==============================
def deduplicate(samples):
    seen = set()
    unique = []

    for s in samples:
        h = md5(s.encode()).hexdigest()
        if h not in seen:
            seen.add(h)
            unique.append(s)

    return unique


# ==============================
# 5. SAVE DATASET
# ==============================
def save_samples(samples, output_file):
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, "w", encoding="utf-8") as f:
        for s in samples:
            f.write(s.strip())
            f.write("\n\n# ===== SAMPLE SEPARATOR =====\n\n")


# ==============================
# MAIN PIPELINE
# ==============================
def process():
    print("Loading data...")

    if not os.path.exists(INPUT_FILE):
        print(f"❌ File not found: {INPUT_FILE}")
        return

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        raw_code = f.read()

    print("Removing noise...")
    clean_code = remove_noise(raw_code)

    print("Extracting structured blocks...")
    samples = extract_code_blocks(clean_code)

    print(f"Total samples before filtering: {len(samples)}")

    print("Filtering broken/useless samples...")
    samples = filter_samples(samples)

    print(f"After filtering: {len(samples)}")

    print("Deduplicating...")
    samples = deduplicate(samples)

    print(f"After deduplication: {len(samples)}")

    print("Saving dataset...")
    save_samples(samples, OUTPUT_FILE)

    print("Done")


if __name__ == "__main__":
    process()

Loading data...
Removing noise...
Extracting structured blocks...
Total samples before filtering: 173305
Filtering broken/useless samples...
After filtering: 126596
Deduplicating...
After deduplication: 123120
Saving dataset...
Done


In [6]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "clean_code_2.txt")
OUTPUT_FILE1 = os.path.join(BASE_DIR, "data", "processed", "clean_code_2_part1.txt")
OUTPUT_FILE2 = os.path.join(BASE_DIR, "data", "processed", "clean_code_2_part2.txt")

# Step 1: Count total lines
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)

half = total_lines // 2

# Step 2: Split file
with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE1, "w", encoding="utf-8") as f1, \
     open(OUTPUT_FILE2, "w", encoding="utf-8") as f2:
    
    for i, line in enumerate(fin):
        if i < half:
            f1.write(line)
        else:
            f2.write(line)

print(f"Done! Split into:\n{OUTPUT_FILE1}\n{OUTPUT_FILE2}")

Done! Split into:
d:\Skills\Machine_Learning\GPT\data\processed\clean_code_2_part1.txt
d:\Skills\Machine_Learning\GPT\data\processed\clean_code_2_part2.txt


In [7]:
# ==============================
# PATH SETUP (YOUR FORMAT)
# ==============================
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "clean_code_2.txt")
OUTPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "cleaned_dataset.txt")

SEPARATOR = "# ===== SAMPLE SEPARATOR ====="

MIN_LINES = 3
MAX_LINES = 200


# ==============================
# HELPERS
# ==============================
def remove_comments(code: str) -> str:
    """
    Remove everything after # (including #)
    """
    cleaned_lines = []
    for line in code.split("\n"):
        line = re.sub(r"#.*", "", line)
        if line.strip():
            cleaned_lines.append(line.rstrip())
    return "\n".join(cleaned_lines)


def is_valid_snippet(code: str) -> bool:
    lines = [l for l in code.split("\n") if l.strip()]

    if len(lines) < MIN_LINES or len(lines) > MAX_LINES:
        return False

    joined = " ".join(lines)

    # Keep meaningful Python structures
    if not any(k in joined for k in ["def ", "class ", "return", "="]):
        return False

    # Remove import-only snippets
    if all(l.strip().startswith(("import ", "from ")) for l in lines):
        return False

    return True


# ==============================
# MAIN PIPELINE
# ==============================
def process():
    print("Loading file...")

    with open(INPUT_FILE, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    print("Splitting samples...")
    samples = text.split(SEPARATOR)

    cleaned_samples = []
    total = len(samples)

    for i, sample in enumerate(samples):
        sample = remove_comments(sample).strip()

        if is_valid_snippet(sample):
            cleaned_samples.append(sample)

        if i % 1000 == 0:
            print(f"Processed {i}/{total}")

    print("Writing output...")

    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for s in cleaned_samples:
            f.write(s + "\n\n" + SEPARATOR + "\n\n")

    print("===================================")
    print(f"Total samples: {total}")
    print(f"Cleaned samples: {len(cleaned_samples)}")
    print("Done")


# ==============================
# ENTRY POINT
# ==============================
if __name__ == "__main__":
    process()

Loading file...
Splitting samples...
Processed 0/159552
Processed 1000/159552
Processed 2000/159552
Processed 3000/159552
Processed 4000/159552
Processed 5000/159552
Processed 6000/159552
Processed 7000/159552
Processed 8000/159552
Processed 9000/159552
Processed 10000/159552
Processed 11000/159552
Processed 12000/159552
Processed 13000/159552
Processed 14000/159552
Processed 15000/159552
Processed 16000/159552
Processed 17000/159552
Processed 18000/159552
Processed 19000/159552
Processed 20000/159552
Processed 21000/159552
Processed 22000/159552
Processed 23000/159552
Processed 24000/159552
Processed 25000/159552
Processed 26000/159552
Processed 27000/159552
Processed 28000/159552
Processed 29000/159552
Processed 30000/159552
Processed 31000/159552
Processed 32000/159552
Processed 33000/159552
Processed 34000/159552
Processed 35000/159552
Processed 36000/159552
Processed 37000/159552
Processed 38000/159552
Processed 39000/159552
Processed 40000/159552
Processed 41000/159552
Processed 

In [8]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "cleaned_dataset.txt")
OUTPUT_FILE1 = os.path.join(BASE_DIR, "data", "processed", "cleaned_dataset_part1.txt")
OUTPUT_FILE2 = os.path.join(BASE_DIR, "data", "processed", "cleaned_dataset_part2.txt")

# Step 1: Count total lines
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)

half = total_lines // 2

# Step 2: Split file
with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE1, "w", encoding="utf-8") as f1, \
     open(OUTPUT_FILE2, "w", encoding="utf-8") as f2:
    
    for i, line in enumerate(fin):
        if i < half:
            f1.write(line)
        else:
            f2.write(line)

print(f"Done! Split into:\n{OUTPUT_FILE1}\n{OUTPUT_FILE2}")

Done! Split into:
d:\Skills\Machine_Learning\GPT\data\processed\cleaned_dataset_part1.txt
d:\Skills\Machine_Learning\GPT\data\processed\cleaned_dataset_part2.txt


In [9]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "cleaned_dataset.txt")
OUTPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "cleaned_dataset_2.txt")

MIN_LINES = 5
MAX_LINES = 80

def is_valid_block(code):
    lines = code.strip().split("\n")

    if len(lines) < MIN_LINES or len(lines) > MAX_LINES:
        return False

    if not (code.strip().startswith("def ") or code.strip().startswith("class ")):
        return False

    if code.strip().endswith(":"):
        return False

    if code.count("import ") > 3:
        return False

    low_signal_keywords = [
        "logger", "logging", "print(", "debug", "warning",
        "to_string", "__repr__", "__str__", "pass"
    ]
    if any(k in code for k in low_signal_keywords):
        return False

    framework_noise = [
        "paddle.", "tensorflow", "_C_ops", "process_group",
        "distributed", "cuda", "xpu"
    ]
    if sum(k in code for k in framework_noise) >= 2:
        return False

    return True


def clean_dataset():
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = f.read()

    blocks = data.split("# ===== SAMPLE SEPARATOR =====")

    cleaned = []
    seen = set()

    for block in blocks:
        code = block.strip()
        if not code:
            continue

        if not is_valid_block(code):
            continue

        key = re.sub(r"\s+", "", code)
        if key in seen:
            continue
        seen.add(key)

        cleaned.append(code)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write("\n\n# ===== SAMPLE SEPARATOR =====\n\n".join(cleaned))

    print(f"Final samples: {len(cleaned)}")


if __name__ == "__main__":
    clean_dataset()

Final samples: 107703


In [10]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "cleaned_dataset_2.txt")
OUTPUT_FILE1 = os.path.join(BASE_DIR, "data", "processed", "cleaned_dataset_2_part1.txt")
OUTPUT_FILE2 = os.path.join(BASE_DIR, "data", "processed", "cleaned_dataset_2_part2.txt")

# Step 1: Count total lines
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)

half = total_lines // 2

# Step 2: Split file
with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE1, "w", encoding="utf-8") as f1, \
     open(OUTPUT_FILE2, "w", encoding="utf-8") as f2:
    
    for i, line in enumerate(fin):
        if i < half:
            f1.write(line)
        else:
            f2.write(line)

print(f"Done! Split into:\n{OUTPUT_FILE1}\n{OUTPUT_FILE2}")

Done! Split into:
d:\Skills\Machine_Learning\GPT\data\processed\cleaned_dataset_2_part1.txt
d:\Skills\Machine_Learning\GPT\data\processed\cleaned_dataset_2_part2.txt


In [11]:
# ===== PATH SETUP =====
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "cleaned_dataset_2.txt")
OUTPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "filtered_dataset.txt")

# ===== CONFIG =====
MIN_LINES = 8
MAX_LINES = 300
MIN_CHARS = 120

LOW_VALUE_PATTERNS = [
    r"^\s*pass\s*$",
    r"^\s*return\s+[a-zA-Z_][a-zA-Z0-9_]*\s*$",
    r"^\s*def __init__\(self\):\s*$",
    r"^\s*class\s+\w+:\s*$",
]

KEYWORDS = [
    "for ", "while ", "if ", "try:", "except", "with ",
    "return ", "import ", "class ", "def "
]


def is_low_value(code: str) -> bool:
    lines = code.strip().split("\n")

    if len(lines) < MIN_LINES or len(lines) > MAX_LINES:
        return True

    if len(code) < MIN_CHARS:
        return True

    for pattern in LOW_VALUE_PATTERNS:
        if re.fullmatch(pattern, code.strip()):
            return True

    if not any(k in code for k in KEYWORDS):
        return True

    non_empty = [l for l in lines if l.strip() and not l.strip().startswith("#")]
    if len(non_empty) < len(lines) * 0.4:
        return True

    return False


def normalize(code: str) -> str:
    code = "\n".join([line.rstrip() for line in code.split("\n")])
    code = re.sub(r"\n{3,}", "\n\n", code)
    return code.strip()


def process():
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = f.read()

    samples = data.split("# ===== SAMPLE SEPARATOR =====")

    kept = []
    seen = set()

    total = len(samples)

    for i, sample in enumerate(samples):
        code = normalize(sample)

        if not code or is_low_value(code):
            continue

        h = hash(code)
        if h in seen:
            continue
        seen.add(h)

        kept.append(code)

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{total} | Kept: {len(kept)}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write("\n\n# ===== SAMPLE SEPARATOR =====\n\n".join(kept))

    print("\nDone")
    print(f"Final samples: {len(kept)}")
    print(f"Reduction: {total} → {len(kept)}")


if __name__ == "__main__":
    process()

Processed 15000/107703 | Kept: 9744
Processed 20000/107703 | Kept: 12797
Processed 35000/107703 | Kept: 22120
Processed 45000/107703 | Kept: 28137
Processed 55000/107703 | Kept: 34912
Processed 65000/107703 | Kept: 41270
Processed 70000/107703 | Kept: 44761
Processed 85000/107703 | Kept: 54620
Processed 90000/107703 | Kept: 58148
Processed 100000/107703 | Kept: 65256
Processed 105000/107703 | Kept: 68782

Done
Final samples: 70948
Reduction: 107703 → 70948


In [12]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "filtered_dataset.txt")
OUTPUT_FILE1 = os.path.join(BASE_DIR, "data", "processed", "filtered_dataset_part1.txt")
OUTPUT_FILE2 = os.path.join(BASE_DIR, "data", "processed", "filtered_dataset_part2.txt")

# Step 1: Count total lines
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)

half = total_lines // 2

# Step 2: Split file
with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE1, "w", encoding="utf-8") as f1, \
     open(OUTPUT_FILE2, "w", encoding="utf-8") as f2:
    
    for i, line in enumerate(fin):
        if i < half:
            f1.write(line)
        else:
            f2.write(line)

print(f"Done! Split into:\n{OUTPUT_FILE1}\n{OUTPUT_FILE2}")

Done! Split into:
d:\Skills\Machine_Learning\GPT\data\processed\filtered_dataset_part1.txt
d:\Skills\Machine_Learning\GPT\data\processed\filtered_dataset_part2.txt


In [13]:
# ===== PATHS =====
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "filtered_dataset.txt")
OUTPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "filtered_dataset_2.txt")

SEPARATOR = "# ===== SAMPLE SEPARATOR ====="

# ===== NOISE RULES =====
TEST_KEYWORDS = ["pytest", "assert", "fixture", "unittest", "TestCase"]
FRAMEWORK_NOISE = ["tf.", "tensorflow", "keras", "torchvision.ops"]
LOW_SIGNAL_IMPORTS = ["import pytest"]

# detect incomplete fragments
INCOMPLETE_PATTERNS = [
    r"^def __init__\(self.*\):\s*$",
    r"^def forward\(self.*\):\s*$",
]

# repeated DL blocks
DUPLICATE_HINTS = [
    "ResNet", "BasicBlock", "Bottleneck", "PreAct"
]


def is_test_code(code: str) -> bool:
    return any(k in code for k in TEST_KEYWORDS)


def is_framework_noise(code: str) -> bool:
    # remove only if it's mostly framework-heavy and not much logic
    count = sum(1 for k in FRAMEWORK_NOISE if k in code)
    return count >= 3


def is_low_signal_import(code: str) -> bool:
    return any(k in code for k in LOW_SIGNAL_IMPORTS)


def is_incomplete(code: str) -> bool:
    lines = code.strip().split("\n")
    if len(lines) <= 3:
        return True

    first = lines[0].strip()

    for pattern in INCOMPLETE_PATTERNS:
        if re.match(pattern, first):
            return True

    # no return / no logic
    if "return" not in code and "for " not in code and "if " not in code:
        return True

    return False


def is_duplicate_heavy(code: str) -> bool:
    return any(hint in code for hint in DUPLICATE_HINTS)


def normalize(code: str) -> str:
    code = "\n".join(line.rstrip() for line in code.split("\n"))
    code = re.sub(r"\n{3,}", "\n\n", code)
    return code.strip()


def process():
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = f.read()

    samples = data.split(SEPARATOR)

    kept = []
    seen = set()

    total = len(samples)

    for i, sample in enumerate(samples):
        code = normalize(sample)

        if not code:
            continue

        if is_test_code(code):
            continue

        if is_low_signal_import(code):
            continue

        if is_incomplete(code):
            continue

        if is_framework_noise(code):
            continue

        # light duplicate filtering
        h = hash(code)
        if h in seen:
            continue
        seen.add(h)

        # remove excessive DL repetition (keep some, not all)
        if is_duplicate_heavy(code) and hash(code[:200]) in seen:
            continue

        kept.append(code)

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{total} | Kept: {len(kept)}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(f"\n\n{SEPARATOR}\n\n".join(kept))

    print("\nDone ✅")
    print(f"Final samples: {len(kept)}")
    print(f"Reduction: {total} → {len(kept)}")


if __name__ == "__main__":
    process()

Processed 10000/70948 | Kept: 7127
Processed 15000/70948 | Kept: 10723
Processed 20000/70948 | Kept: 13941
Processed 30000/70948 | Kept: 20280
Processed 35000/70948 | Kept: 23647
Processed 45000/70948 | Kept: 27006
Processed 50000/70948 | Kept: 30828
Processed 60000/70948 | Kept: 36319
Processed 65000/70948 | Kept: 39385
Processed 70000/70948 | Kept: 42665

Done ✅
Final samples: 43176
Reduction: 70948 → 43176


In [15]:
# ===== PATHS =====
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "filtered_dataset_2.txt")
OUTPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "filtered_dataset_3.txt")

SEPARATOR = "# ===== SAMPLE SEPARATOR ====="

# ===== CONFIG =====
MIN_LINES = 10

BANNED_PREFIXES = ("get_", "fix_", "test_")
BANNED_IMPORTS = ("art.", "sklearn", "torchvision")


def normalize(code: str) -> str:
    code = "\n".join(line.rstrip() for line in code.split("\n"))
    code = re.sub(r"\n{3,}", "\n\n", code)
    return code.strip()


def has_banned_function(code: str) -> bool:
    for line in code.split("\n"):
        line = line.strip()
        if line.startswith("def "):
            name = line.split("(")[0].replace("def ", "")
            if name.startswith(BANNED_PREFIXES):
                return True
    return False


def has_heavy_imports(code: str) -> bool:
    count = sum(1 for imp in BANNED_IMPORTS if imp in code)
    return count >= 2


def is_tiny(code: str) -> bool:
    lines = [l for l in code.split("\n") if l.strip() and not l.strip().startswith("#")]
    return len(lines) < MIN_LINES


def process():
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = f.read()

    samples = data.split(SEPARATOR)

    kept = []
    seen = set()
    total = len(samples)

    for i, sample in enumerate(samples):
        code = normalize(sample)

        if not code:
            continue

        if has_banned_function(code):
            continue

        if has_heavy_imports(code):
            continue

        if is_tiny(code):
            continue

        h = hash(code)
        if h in seen:
            continue
        seen.add(h)

        kept.append(code)

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{total} | Kept: {len(kept)}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(f"\n\n{SEPARATOR}\n\n".join(kept))

    print("\nDone")
    print(f"Final samples: {len(kept)}")
    print(f"Reduction: {total} → {len(kept)}")


if __name__ == "__main__":
    process()

Processed 10000/43176 | Kept: 6847
Processed 15000/43176 | Kept: 10311
Processed 20000/43176 | Kept: 13909
Processed 25000/43176 | Kept: 17287
Processed 30000/43176 | Kept: 20914
Processed 40000/43176 | Kept: 28080

Done
Final samples: 30524
Reduction: 43176 → 30524


In [16]:
# ===== PATHS =====
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "filtered_dataset_3.txt")
OUTPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "filtered_dataset_4.txt")

SEPARATOR = "# ===== SAMPLE SEPARATOR ====="

# ===== STRICT FILTER CONFIG =====
BANNED_IMPORTS = (
    "sklearn", "torchvision", "tensorflow", "keras",
    "art.", "cv2", "pandas", "datasets", "qdrant",
    "weaviate", "openai", "agent", "TriggerFlow"
)

BANNED_KEYWORDS = (
    "Dataset", "DataLoader", "Classifier", "Postprocessor",
    "Scheduler", "Workflow", "Agent", "Knowledge",
    "fit(", "predict(", "transform(", "load_"
)

MIN_LINES = 12
MAX_LINES = 200


def normalize(code: str) -> str:
    code = "\n".join(line.rstrip() for line in code.split("\n"))
    code = re.sub(r"\n{3,}", "\n\n", code)
    return code.strip()


def has_banned_imports(code: str) -> bool:
    return any(bad in code for bad in BANNED_IMPORTS)


def has_banned_keywords(code: str) -> bool:
    return any(bad in code for bad in BANNED_KEYWORDS)


def is_valid_structure(code: str) -> bool:
    lines = code.split("\n")

    if len(lines) < MIN_LINES or len(lines) > MAX_LINES:
        return False

    # must contain meaningful logic
    logic_tokens = ["for ", "while ", "if ", "return ", "try:", "with "]
    if not any(tok in code for tok in logic_tokens):
        return False

    # avoid too many imports
    import_count = sum(1 for l in lines if l.strip().startswith("import") or l.strip().startswith("from"))
    if import_count > 3:
        return False

    return True


def process():
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = f.read()

    samples = data.split(SEPARATOR)

    kept = []
    seen = set()

    total = len(samples)

    for i, sample in enumerate(samples):
        code = normalize(sample)

        if not code:
            continue

        if has_banned_imports(code):
            continue

        if has_banned_keywords(code):
            continue

        if not is_valid_structure(code):
            continue

        h = hash(code)
        if h in seen:
            continue
        seen.add(h)

        kept.append(code)

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{total} | Kept: {len(kept)}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(f"\n\n{SEPARATOR}\n\n".join(kept))

    print("\nDone ✅")
    print(f"Final samples: {len(kept)}")
    print(f"Reduction: {total} → {len(kept)}")


if __name__ == "__main__":
    process()

Processed 5000/30524 | Kept: 3596
Processed 10000/30524 | Kept: 7050
Processed 15000/30524 | Kept: 10613
Processed 20000/30524 | Kept: 14412
Processed 25000/30524 | Kept: 18189
Processed 30000/30524 | Kept: 21779

Done ✅
Final samples: 21939
Reduction: 30524 → 21939


In [18]:
# ===== PATHS =====
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "filtered_dataset_4.txt")
OUTPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "shuffled_dataset.txt")

SEPARATOR = "# ===== SAMPLE SEPARATOR ====="


def process():
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = f.read()

    # split into blocks
    samples = [s.strip() for s in data.split(SEPARATOR) if s.strip()]

    print(f"Total samples before shuffle: {len(samples)}")

    # shuffle in-place
    random.shuffle(samples)

    # write back
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(f"\n\n{SEPARATOR}\n\n".join(samples))

    print("Done")
    print(f"Shuffled samples: {len(samples)}")


if __name__ == "__main__":
    process()

Total samples before shuffle: 21939
Done
Shuffled samples: 21939


In [19]:
# ===== PATHS =====
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

INPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "shuffled_dataset.txt")

OUTPUT_DIR = os.path.join(BASE_DIR, "data", "dataset")
TRAIN_FILE = os.path.join(OUTPUT_DIR, "train.txt")
VAL_FILE = os.path.join(OUTPUT_DIR, "val.txt")
TEST_FILE = os.path.join(OUTPUT_DIR, "test.txt")

SEPARATOR = "# ===== SAMPLE SEPARATOR ====="

# ===== SPLIT RATIOS =====
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1


def process():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = f.read()

    samples = [s.strip() for s in data.split(SEPARATOR) if s.strip()]

    print(f"Total samples: {len(samples)}")

    # shuffle before split
    random.seed(42)
    random.shuffle(samples)

    total = len(samples)
    train_end = int(total * TRAIN_RATIO)
    val_end = train_end + int(total * VAL_RATIO)

    train_samples = samples[:train_end]
    val_samples = samples[train_end:val_end]
    test_samples = samples[val_end:]

    # write files
    with open(TRAIN_FILE, "w", encoding="utf-8") as f:
        f.write(f"\n\n{SEPARATOR}\n\n".join(train_samples))

    with open(VAL_FILE, "w", encoding="utf-8") as f:
        f.write(f"\n\n{SEPARATOR}\n\n".join(val_samples))

    with open(TEST_FILE, "w", encoding="utf-8") as f:
        f.write(f"\n\n{SEPARATOR}\n\n".join(test_samples))

    print("Done")
    print(f"Train: {len(train_samples)}")
    print(f"Val: {len(val_samples)}")
    print(f"Test: {len(test_samples)}")


if __name__ == "__main__":
    process()

Total samples: 21939
Done
Train: 17551
Val: 2193
Test: 2195
